
# Multimodal Training – Tabular + Satellite Images

This notebook demonstrates training of a CNN + Tabular fusion model
and evaluates it using RMSE and R².


In [ ]:

import pandas as pd
import numpy as np
import torch

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.config import cfg
from src.datasets import HouseDataset
from src.model import FusionModel
from src.train import train_one_epoch, evaluate
from src.data_fetcher import download


In [ ]:

# Load and prepare data
df = pd.read_excel(cfg.train_xlsx)
df['price'] = np.log1p(df['price'])

scaler = StandardScaler()
scaler.fit(df[cfg.tab_feats])

train_df, val_df = train_test_split(
    df, test_size=cfg.val_split, random_state=cfg.seed
)


In [ ]:

# Fetch images
img_paths = download(df)


In [ ]:

# Datasets and loaders
train_ds = HouseDataset(train_df, img_paths, scaler, train=True)
val_ds = HouseDataset(val_df, img_paths, scaler, train=True)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)


In [ ]:

# Model
model = FusionModel(tab_in=len(cfg.tab_feats)).to(cfg.device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay
)
criterion = torch.nn.SmoothL1Loss()


In [ ]:

# Training loop
for epoch in range(cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_rmse, val_r2 = evaluate(model, val_loader, criterion)

    print(
        f"Epoch {epoch+1}/{cfg.epochs} | "
        f"RMSE: {val_rmse:.2f} | R2: {val_r2:.3f}"
    )



### Conclusion
- Multimodal model improves performance over tabular-only baseline
- Visual features capture neighborhood characteristics
